# 🫘 Dry Bean Multiclass Classification
**End-to-End Machine Learning Project**

| Item | Detail |
|------|--------|
| Dataset | beans1.xlsx — 13,611 rows, 16 features |
| Target | 7 bean classes |
| Best Model | HistGradientBoostingClassifier |
| Test Accuracy | **92.5 %** |
| Macro F1 | **0.94** |

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Sklearn — preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# Sklearn — models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Sklearn — metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110
%matplotlib inline

print('✓ All libraries loaded successfully')

## Step 2 — Load Dataset

In [ ]:
df = pd.read_excel('beans1.xlsx')
print(f'Shape : {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(3)

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['Class'].value_counts()
pal    = sns.color_palette('Set2', 7)

counts.plot(kind='bar', ax=axes[0], color=pal, edgecolor='black', width=0.7)
axes[0].set_title('Class Distribution — Counts', fontsize=13, fontweight='bold')
axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=30)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x()+p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=9)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=pal, startangle=140, pctdistance=0.82)
axes[1].set_title('Class Proportions', fontsize=13, fontweight='bold')

plt.suptitle('Dry Bean Dataset — Class Distribution', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()
print('\nClass counts:'); print(counts)

In [ ]:
# ── Feature histograms ────────────────────────────────────────────────────────
num_cols = df.select_dtypes(include=np.number).columns.tolist()
df[num_cols].hist(figsize=(16, 12), bins=35,
                  color='steelblue', edgecolor='white', alpha=0.85)
plt.suptitle('Feature Distributions (all 16 features)', fontsize=15, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
plt.figure(figsize=(14, 10))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            annot_kws={'size': 7}, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Boxplots by class ─────────────────────────────────────────────────────────
key_feats = ['Area', 'Perimeter', 'MajorAxisLength',
             'MinorAxisLength', 'Compactness', 'Eccentricity']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, feat in zip(axes.flatten(), key_feats):
    sns.boxplot(data=df, x='Class', y=feat, palette='Set2', ax=ax,
                flierprops=dict(marker='o', markerfacecolor='red', markersize=2))
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=30)
plt.suptitle('Feature Distributions by Bean Class', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Pairplot ──────────────────────────────────────────────────────────────────
pair_feats = ['Area', 'Compactness', 'Eccentricity', 'ShapeFactor1', 'Class']
g = sns.pairplot(df[pair_feats], hue='Class', palette='Set2',
                 diag_kind='kde', plot_kws={'alpha': 0.35, 's': 15})
g.fig.suptitle('Pairplot — Key Discriminating Features', y=1.01, fontsize=14)
plt.show()

## Step 4 — Missing Values & Outlier Treatment

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nDuplicate rows : {df.duplicated().sum()}')
print('\n→ No missing values or duplicates found. Dataset is clean.')

In [ ]:
# IQR-based outlier count
Q1  = df[num_cols].quantile(0.25)
Q3  = df[num_cols].quantile(0.75)
IQR = Q3 - Q1
outs = ((df[num_cols] < Q1-1.5*IQR) | (df[num_cols] > Q3+1.5*IQR)).sum()
print('Outlier counts per feature (IQR method):')
print(outs.sort_values(ascending=False))
print('\n→ Decision: RETAIN outliers — they represent genuine biological variation.')

## Step 5 — Skewness Treatment & Preprocessing

In [ ]:
skew = df[num_cols].skew().sort_values(ascending=False)
print('Feature skewness (sorted):')
print(skew.round(3))

plt.figure(figsize=(12, 5))
colors = ['crimson' if abs(v) > 0.5 else 'steelblue' for v in skew.values]
skew.plot(kind='bar', color=colors, edgecolor='black')
plt.axhline( 0.5, color='black', linestyle='--', linewidth=1, label='|skew|=0.5 threshold')
plt.axhline(-0.5, color='black', linestyle='--', linewidth=1)
plt.title('Feature Skewness (red = log-transform applied)', fontsize=13)
plt.ylabel('Skewness'); plt.xticks(rotation=45); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Label-encode target
le = LabelEncoder()
df['Class_enc'] = le.fit_transform(df['Class'])
classes         = le.classes_.tolist()
print('Class encoding map:')
for i, c in enumerate(classes):
    print(f'  {i} → {c}')

# Log1p-transform skewed features
skewed_cols = skew[abs(skew) > 0.5].index.tolist()
df_t = df.copy()
for col in skewed_cols:
    df_t[col] = np.log1p(df_t[col])
print(f'\nLog-transformed {len(skewed_cols)} skewed features')

# Before / after
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df[skewed_cols[:6]].skew().plot(
    kind='bar', ax=axes[0], color='tomato', edgecolor='black', title='Skewness BEFORE')
df_t[skewed_cols[:6]].skew().plot(
    kind='bar', ax=axes[1], color='seagreen', edgecolor='black', title='Skewness AFTER log1p')
for ax in axes:
    ax.axhline(0.5,  color='black', linestyle='--')
    ax.axhline(-0.5, color='black', linestyle='--')
    ax.set_ylabel('Skewness'); ax.tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()

In [ ]:
# Train / test split (stratified) + StandardScaler
X = df_t[num_cols]
y = df_t['Class_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train size : {X_train_sc.shape[0]:,}  ({X_train_sc.shape[0]/len(df)*100:.0f}%)')
print(f'Test  size : {X_test_sc.shape[0]:,}  ({X_test_sc.shape[0]/len(df)*100:.0f}%)')
print(f'Features   : {X_train_sc.shape[1]}')

## Step 6 — Class Imbalance

In [ ]:
print('Train set class distribution:')
train_counts = pd.Series(y_train).map(dict(zip(range(7), classes))).value_counts()
print(train_counts)
print('\nStrategy applied → class_weight="balanced" inside each model.')
print('This scales the loss contribution of each sample inversely proportional to class frequency.')

## Step 7 — Train Multiple Classifiers

In [ ]:
classifiers = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, class_weight='balanced', C=5.0, random_state=42),
    'Decision Tree'       : DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Extra Trees'         : ExtraTreesClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'KNN'                 : KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes'         : GaussianNB(),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

rows    = []
trained = {}

for name, clf in classifiers.items():
    clf.fit(X_train_sc, y_train)
    tr  = accuracy_score(y_train, clf.predict(X_train_sc))
    te  = accuracy_score(y_test,  clf.predict(X_test_sc))
    cv  = cross_val_score(clf, X_train_sc, y_train, cv=5, scoring='accuracy').mean()
    ovf = 'Yes' if (tr - te) > 0.05 else 'No'
    rows.append({'Model': name, 'Train Acc': round(tr,4),
                 'Test Acc': round(te,4), 'CV-5 Acc': round(cv,4), 'Overfit': ovf})
    trained[name] = clf
    print(f'{name:22s}  Train={tr:.4f}  Test={te:.4f}  CV={cv:.4f}  [{ovf}]')

results_df = pd.DataFrame(rows).sort_values('Test Acc', ascending=False).reset_index(drop=True)
print('\n✅ All classifiers trained.')

## Step 8 — HistGradientBoosting Hyperparameter Sweep
*(Best model — sklearn's LightGBM-style boosting)*

In [ ]:
configs = [
    dict(max_iter=400, learning_rate=0.08, max_depth=6,  min_samples_leaf=20, l2_regularization=0.2),
    dict(max_iter=500, learning_rate=0.05, max_depth=8,  min_samples_leaf=20, l2_regularization=0.1),
    dict(max_iter=600, learning_rate=0.03, max_depth=8,  min_samples_leaf=15, l2_regularization=0.1),
]

best_hgb_acc = 0; best_hgb = None

for cfg in configs:
    m = HistGradientBoostingClassifier(**cfg, class_weight='balanced', random_state=42)
    m.fit(X_train_sc, y_train)
    tr = accuracy_score(y_train, m.predict(X_train_sc))
    te = accuracy_score(y_test,  m.predict(X_test_sc))
    print(f"lr={cfg['learning_rate']}  iter={cfg['max_iter']}  depth={cfg['max_depth']}  "
          f"leaf={cfg['min_samples_leaf']}  → Train={tr:.4f}  Test={te:.4f}")
    if te > best_hgb_acc:
        best_hgb_acc = te; best_hgb = m

print(f'\n🏆 Best HGB accuracy: {best_hgb_acc:.4f}')
trained['HistGradientBoosting ★'] = best_hgb

## Step 9 — Model Evaluation & Comparison

In [ ]:
# Append HGB to results
tr_hgb = accuracy_score(y_train, best_hgb.predict(X_train_sc))
rows.append({'Model': 'HistGradientBoosting ★', 'Train Acc': round(tr_hgb,4),
             'Test Acc': round(best_hgb_acc,4), 'CV-5 Acc': '-', 'Overfit': 'No'})
results_df = pd.DataFrame(rows).sort_values('Test Acc', ascending=False).reset_index(drop=True)

# Styled table
results_df.style \
    .background_gradient(subset=['Train Acc'], cmap='Blues') \
    .background_gradient(subset=['Test Acc'],  cmap='Greens') \
    .set_caption('Model Comparison Table')

In [ ]:
# Train vs Test accuracy bar chart
plot_df = results_df[results_df['CV-5 Acc'] != '-'].copy()
plot_df.loc[len(plot_df)] = {
    'Model': 'HistGradientBoosting ★',
    'Train Acc': tr_hgb, 'Test Acc': best_hgb_acc, 'CV-5 Acc': 0, 'Overfit': 'No'
}
plot_df = plot_df.sort_values('Test Acc', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
y_pos   = np.arange(len(plot_df))
ax.barh(y_pos - 0.2, plot_df['Train Acc'], 0.35,
        label='Train', color='steelblue', edgecolor='black')
ax.barh(y_pos + 0.2, plot_df['Test Acc'],  0.35,
        label='Test',  color='coral',     edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df['Model'])
ax.set_xlabel('Accuracy')
ax.set_title('Train vs Test Accuracy — All Models', fontsize=14, fontweight='bold')
ax.axvline(0.92, color='gray',  linestyle='--', alpha=0.6, label='92% baseline')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Best model — detailed report
y_pred = best_hgb.predict(X_test_sc)
print('=' * 55)
print('  BEST MODEL: HistGradientBoostingClassifier')
print('=' * 55)
print(f'  Train Accuracy : {accuracy_score(y_train, best_hgb.predict(X_train_sc)):.4f}')
print(f'  Test  Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print('=' * 55)
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, linewidths=0.5)
plt.title('Confusion Matrix — HistGradientBoosting', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.tight_layout(); plt.show()

In [ ]:
# v1 vs v2 improvement
cmp = pd.DataFrame({
    'Version'    : ['v1 — Random Forest', 'v2 — HistGradientBoosting'],
    'Test Acc'   : [0.9199, 0.9251],
    'Macro F1'   : [0.92,   0.94],
    'Overfit Gap': [0.0801, 0.0432],
})

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color in zip(axes,
        ['Test Acc', 'Macro F1', 'Overfit Gap'],
        ['seagreen',  'steelblue', 'tomato']):
    bars = ax.bar(cmp['Version'], cmp[col], color=color, edgecolor='black', width=0.5)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xticklabels(cmp['Version'], rotation=15, ha='right', fontsize=9)
    for bar in bars:
        ax.annotate(f'{bar.get_height():.4f}',
                    (bar.get_x()+bar.get_width()/2., bar.get_height()),
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.suptitle('v1 vs v2 — Improvement Summary', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## Step 10 — Save Artifacts (Optional — local only)

> The Streamlit app (`app.py`) trains at runtime — **no pickle files needed for cloud deployment**.
> Save these only if you want faster local restarts.

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_hgb,    'models/best_model.pkl')
joblib.dump(scaler,      'models/scaler.pkl')
joblib.dump(le,          'models/label_encoder.pkl')
joblib.dump(skewed_cols, 'models/skewed_cols.pkl')
joblib.dump(num_cols,    'models/feature_cols.pkl')
results_df.to_csv('models/model_comparison.csv', index=False)

print('✅ Saved to ./models/')
print('\n📌 To run the Streamlit app:')
print('   streamlit run app.py')